In [1]:
import site
import sys
import subprocess

# 1. Registrar la ruta de paquetes del usuario en el sistema
user_site = '/headless/.local/lib/python3.12/site-packages'
site.addsitedir(user_site)

# 2. Intentar importar; si falla, instalarlo automáticamente
try:
    from PyLTSpice import RawRead
    print("¡PyLTSpice ya estaba instalado y listo para usar!")
except ImportError:
    print("PyLTSpice no fue encontrado. Instalando automáticamente...")
    
    # Ejecuta pip install dentro del mismo Kernel de Python
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--user", "PyLTSpice"])
    
    # Re-asegurar que la ruta esté cargada
    site.addsitedir(user_site)
    
    # Importación tras la instalación
    from PyLTSpice import RawRead
    print("¡PyLTSpice se instaló e importó correctamente!")

¡PyLTSpice ya estaba instalado y listo para usar!


In [2]:

import subprocess
from PyLTSpice import RawRead

# 1. Definir las rutas (reemplaza con tus directorios exactos)
netlist_path = "/headless/.xschem/simulations/ldo_tb.spice"
raw_path = "/foss/designs/LDO-sky130/xschem/spice_raw_log/raw/"


# 2. Configurar y ejecutar el comando de Ngspice
# -b: Ejecuta en modo batch (sin interfaz gráfica)
# -r: Define el archivo de salida raw
# -o: Define el archivo log para atrapar errores de convergencia o sintaxis
comando = ['/foss/tools/ngspice/bin/ngspice', "-b", netlist_path]


# Añadir cwd=carpeta_destino al subproceso
resultado = subprocess.run(comando, capture_output=True, text=True, cwd=raw_path)
mi_log = resultado.stdout

# Puedes imprimirlo si hubo algún problema o guardarlo si lo necesitas
if resultado.returncode != 0:
    print("La simulación falló. Aquí está el error de Ngspice:\n")
    print(resultado.stderr)  # Imprime los errores críticos
else:
    print("¡Simulación terminada con éxito!")
    # print(mi_log) # Descomenta esto si quieres ver todo el texto del log en tu consola


¡Simulación terminada con éxito!


In [5]:
import re


# 2. Crear un diccionario vacío para guardar los datos
parametros_ldo = {}

# 3. Expresión regular que busca "cualquier_palabra = numero"
# Soporta decimales (72.8159) y notación científica (1.00669e+06)
patron = r"([a-zA-Z0-9_]+)\s*=\s*([+-]?\d+\.?\d*[eE]?[+-]?\d*)"

# 4. Encontrar todas las coincidencias en el texto de mi_log
coincidencias = re.findall(patron, mi_log)
#print(coincidencias)
# 5. Llenar el diccionario convirtiendo el texto a números reales (float)
for nombre, valor in coincidencias:
    # nombre.strip() quita espacios extra por seguridad
    parametros_ldo[nombre.strip().lower()] = float(valor)

# eliminamos algunos que no sean importantes 
del parametros_ldo["tnom"]
print(parametros_ldo)
# ¡Listo! Ahora puedes usar tus medidas del LDO como variables de Python
print("--- Datos extraídos del LOG ---")
print(f"Margen de fase: {parametros_ldo.get('margen_fase')} grados")
print(f"Ganancia UGB: {parametros_ldo.get('ugb')} Hz")
print(f"Load Regulation: {parametros_ldo.get('load_regulation')}")

# Si quieres ver todo lo que capturó:
# print(parametros_ldo)

{'temp': 125.0, 'ugb': 1014670.0, 'fase_ugb': 67.1489, 'ganancia_banda_plana': 75.8351, 'margen_fase': 67.1489, 'gananciapsrr_1': 58.2289, 'gananciapsrr_1k': 57.6445, 'gananciapsrr_10k': 46.3992, 'gananciapsrr_1m': 28.4111, 'voload_max': 1.19329, 'at': 1.3, 'voload_min': 1.19319, 'load_regulation': 9.7e-05, 'voline_max': 1.19345, 'voline_min': 1.19263, 'line_regulation': 0.000816}
--- Datos extraídos del LOG ---
Margen de fase: 67.1489 grados
Ganancia UGB: 1014670.0 Hz
Load Regulation: 9.7e-05


In [25]:
from PyLTSpice import RawRead
import os
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px


# Cargar el archivo .raw
raw_tran = os.path.join(raw_path,"LDO_tran_picos.raw")

raw = RawRead(os.path.join(raw_path,"LDO_op.raw"))

# Imprimir la lista de todos los nombres de nodos y ramas guardados
print("Variables disponibles:")
for var in raw.get_trace_names():
    print(" -", var)

"""
# Extraer el eje horizontal (Tiempo en .tran, Frecuencia en .ac, Voltaje en .dc)
tiempo = raw.get_trace("time").get_wave()

# Extraer voltajes y corrientes de interés
v_out = raw.get_trace("v(vo)").get_wave()
i_load = raw.get_trace("i(vsense)").get_wave()
# graficar:
fig = px.line(x=tiempo, y=v_out, title="Medicion de undershoot y overshoot en el LDO",labels={'x': 'Tiempo [s]', 'y': 'Voltaje de salida [V]'})
fig.show()
"""

Variables disponibles:
 - v(x1.net1)
 - v(m.x1.x1.xmc2.msky130_fd_pr__nfet_01v8#body)
 - v(m.x1.x1.xmc2.msky130_fd_pr__nfet_01v8#dbody)
 - v(m.x1.x1.xmc2.msky130_fd_pr__nfet_01v8#sbody)
 - v(m.x1.x1.xmn.msky130_fd_pr__nfet_01v8#body)
 - v(m.x1.x1.xmn.msky130_fd_pr__nfet_01v8#dbody)
 - v(m.x1.x1.xmn.msky130_fd_pr__nfet_01v8#sbody)
 - v(m.x1.x1.xmnr.msky130_fd_pr__pfet_01v8#body)
 - v(m.x1.x1.xmnr.msky130_fd_pr__pfet_01v8#dbody)
 - v(m.x1.x1.xmnr.msky130_fd_pr__pfet_01v8#sbody)
 - v(m.x1.x1.xmo.msky130_fd_pr__pfet_01v8#body)
 - v(m.x1.x1.xmo.msky130_fd_pr__pfet_01v8#dbody)
 - v(m.x1.x1.xmo.msky130_fd_pr__pfet_01v8#sbody)
 - v(m.x1.x1.xmp.msky130_fd_pr__nfet_01v8#body)
 - v(m.x1.x1.xmp.msky130_fd_pr__nfet_01v8#dbody)
 - v(m.x1.x1.xmp.msky130_fd_pr__nfet_01v8#sbody)
 - v(m.x1.x1.xmpr.msky130_fd_pr__pfet_01v8#body)
 - v(m.x1.x1.xmpr.msky130_fd_pr__pfet_01v8#dbody)
 - v(m.x1.x1.xmpr.msky130_fd_pr__pfet_01v8#sbody)
 - v(m.x1.x1.xmr.msky130_fd_pr__nfet_01v8#body)
 - v(m.x1.x1.xmr.msky130_fd_pr

IndexError: <spicelib.raw.plot_data.PlotData object at 0x7fda6e7e8ad0> doesn't contain trace "time"
Valid traces are ['v(x1.net1)', 'v(m.x1.x1.xmc2.msky130_fd_pr__nfet_01v8#body)', 'v(m.x1.x1.xmc2.msky130_fd_pr__nfet_01v8#dbody)', 'v(m.x1.x1.xmc2.msky130_fd_pr__nfet_01v8#sbody)', 'v(m.x1.x1.xmn.msky130_fd_pr__nfet_01v8#body)', 'v(m.x1.x1.xmn.msky130_fd_pr__nfet_01v8#dbody)', 'v(m.x1.x1.xmn.msky130_fd_pr__nfet_01v8#sbody)', 'v(m.x1.x1.xmnr.msky130_fd_pr__pfet_01v8#body)', 'v(m.x1.x1.xmnr.msky130_fd_pr__pfet_01v8#dbody)', 'v(m.x1.x1.xmnr.msky130_fd_pr__pfet_01v8#sbody)', 'v(m.x1.x1.xmo.msky130_fd_pr__pfet_01v8#body)', 'v(m.x1.x1.xmo.msky130_fd_pr__pfet_01v8#dbody)', 'v(m.x1.x1.xmo.msky130_fd_pr__pfet_01v8#sbody)', 'v(m.x1.x1.xmp.msky130_fd_pr__nfet_01v8#body)', 'v(m.x1.x1.xmp.msky130_fd_pr__nfet_01v8#dbody)', 'v(m.x1.x1.xmp.msky130_fd_pr__nfet_01v8#sbody)', 'v(m.x1.x1.xmpr.msky130_fd_pr__pfet_01v8#body)', 'v(m.x1.x1.xmpr.msky130_fd_pr__pfet_01v8#dbody)', 'v(m.x1.x1.xmpr.msky130_fd_pr__pf

In [4]:
import pandas as pd
import numpy as np
from PyLTSpice import RawRead
import os
import re
from matplotlib.ticker import EngFormatter  # La librería mágica

def generar_tabla_inteligente_op(ruta_raw):
    """
    Lee un archivo OP, agrupa los datos por transistor, filtra la basura,
    y calcula ro, gm/Id y fT automáticamente.
    """
    if not os.path.exists(ruta_raw):
        print(f"⚠️ El archivo no existe: {ruta_raw}")
        return None
        
    try:
        raw_op = RawRead(ruta_raw)
    except Exception as e:
        print(f"⚠️ Error al leer el archivo RAW: {e}")
        return None

    variables = raw_op.get_trace_names()
    datos_agrupados = {}

    patron = r"x([a-zA-Z0-9_]+)\.msky130.*?\[([a-zA-Z0-9_]+)\]"

    for var in variables:
        match = re.search(patron, var, re.IGNORECASE)
        if match:
            transistor = match.group(1).upper() 
            parametro = match.group(2).lower()  
            
            wave = raw_op.get_trace(var).get_wave()
            if len(wave) > 0:
                valor = float(np.real(wave[0]))
                
                if parametro in ['id', 'vgs', 'vth', 'vdsat']:
                    valor = abs(valor)
                    
                if transistor not in datos_agrupados:
                    datos_agrupados[transistor] = {}
                    
                datos_agrupados[transistor][parametro] = valor

    # ==============================================================================
    # CÁLCULOS INTELIGENTES
    # ==============================================================================
    for t, params in datos_agrupados.items():
        if 'gds' in params and params['gds'] != 0:
            params['ro'] = 1 / params['gds']
            
        if 'gm' in params and 'id' in params and params['id'] != 0:
            params['gm/id'] = params['gm'] / params['id']
            
        if 'gm' in params and 'cgg' in params and params['cgg'] != 0:
            # EL ÚNICO CAMBIO: Le quitamos el "/ 1e9" para que la librería ponga la 'G' sola
            params['ft'] = params['gm'] / (2 * np.pi * params['cgg'])

    df = pd.DataFrame.from_dict(datos_agrupados, orient='index')
    
    # Le quitamos el (GHz) al nombre de la columna
    columnas_ideales = ['id', 'vgs', 'vth', 'vdsat', 'gm', 'ro', 'gm/id', 'cgg', 'ft']
    columnas_finales = [c for c in columnas_ideales if c in df.columns]
    df = df[columnas_finales]
    
    return df
raw_path="/headless/.xschem/simulations/"
# ==============================================================================
# EJEMPLO DE USO:
# ==============================================================================
ruta_op = os.path.join(raw_path, "LDO_op.raw")
tabla_resultados = generar_tabla_inteligente_op(ruta_op)

# Creamos la máscara visual (places=2 son los decimales)
formateador_si = EngFormatter(places=2)

# Se la aplicamos a la tabla de Pandas
tabla_bonita = tabla_resultados.style.format(lambda x: formateador_si(x) if pd.notna(x) else "-")

display(tabla_bonita)